# import all libraries

In [44]:



pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [45]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import numpy as np
from tqdm import tqdm

# SingleRNN

In [46]:
class SingleRNN(nn.Module):
    def __init__(self, n_inputs, n_neurons):
        super(SingleRNN, self).__init__()
        self.Wx= torch.randn(n_inputs, n_neurons)
        self.Wy= torch.randn(n_neurons, n_neurons)
        
        self.b =torch.zeros(1, n_neurons)
        
    def forward(self, X0, X1):
        self.Y0 = torch.tanh(torch.mm(X0, self.Wx) + self.b)
        self.Y1 = torch.tanh(torch.mm(self.Y0, self.Wy)+
                             torch.mm(X1, self.Wx)+ self.b)
        return self.Y0, self.Y1

In [47]:
N_INPUT =4
N_NEURONS = 1
X0_batch = torch.tensor([[0,1,2,0], [3,4,5,0],
                         [6,7,8,0], [9,0,1,0]],
                        dtype = torch.float)
X1_batch = torch.tensor([[9,8,7,0], [0,0,0,0],
                         [6,5,4,0], [3,2,1,9]],
                        dtype = torch.float)
model = SingleRNN(N_INPUT, N_NEURONS)
Y0_val, Y1_val = model(X0_batch, X1_batch)

In [48]:
print(Y0_val)
print(Y1_val)

tensor([[-0.9971],
        [-1.0000],
        [-1.0000],
        [-1.0000]])
tensor([[-1.0000],
        [ 0.6836],
        [-1.0000],
        [ 1.0000]])


# BasicRNN

In [49]:
class BasicRNN(nn.Module):
    def __init__(self, n_inputs, n_neurons):
        super(BasicRNN, self).__init__()
        self.Wx= torch.randn(n_inputs, n_neurons)
        self.Wy= torch.randn(n_neurons, n_neurons)
        
        self.b =torch.zeros(1, n_neurons)
        
    def forward(self, X0, X1):
        self.Y0 = torch.tanh(torch.mm(X0, self.Wx) + self.b)
        self.Y1 = torch.tanh(torch.mm(self.Y0, self.Wy)+
                             torch.mm(X1, self.Wx)+ self.b)
        return self.Y0, self.Y1

In [50]:
N_INPUT =3
N_NEURONS = 5
X0_batch = torch.tensor([[0,1,2], [3,4,5],
                         [6,7,8], [9,0,1]],
                        dtype = torch.float)
X1_batch = torch.tensor([[9,8,7], [0,0,0],
                         [6,5,4], [3,2,1]],
                        dtype = torch.float)
model = BasicRNN(N_INPUT, N_NEURONS)
Y0_val, Y1_val = model(X0_batch, X1_batch)

In [51]:
print(Y0_val)
print(Y1_val)

tensor([[ 0.2740,  0.1272,  0.9965,  0.9951,  0.9734],
        [ 0.9979, -0.8226,  1.0000,  1.0000,  1.0000],
        [ 1.0000, -0.9854,  1.0000,  1.0000,  1.0000],
        [ 1.0000, -1.0000, -1.0000,  1.0000,  1.0000]])
tensor([[ 1.0000, -0.9994,  0.9999,  1.0000,  1.0000],
        [ 0.8706,  0.5110,  0.1430, -0.9960,  0.0860],
        [ 1.0000, -0.9713,  0.9979,  1.0000,  1.0000],
        [ 0.9839,  0.0661,  0.9998,  0.9899,  1.0000]])


# Pytorch built-in RNN cell


In [52]:
rnn = nn.RNNCell(3,5)
X_batch = torch.tensor([[[0,1,2], [3,4,5],
                         [6,7,8], [9,0,1]],
                        [[9,8,7], [0,0,0],
                         [6,5,4],[3,2,1]]
                        ], dtype = torch.float)
hx =torch.randn(4,5)
output = []

for i in range(2):
    hx =rnn(X_batch[i],hx)
    output.append(hx)
print(output)

[tensor([[-0.2164, -0.8989, -0.6751, -0.9025, -0.3535],
        [ 0.2623, -0.9982, -0.6744, -0.8281, -0.7907],
        [ 0.8528, -1.0000, -0.7109, -0.9414, -0.9591],
        [ 0.9973, -0.9903,  0.9111,  0.4720, -0.8866]],
       grad_fn=<TanhBackward0>), tensor([[ 0.9968, -1.0000, -0.6594, -0.7942, -0.9970],
        [ 0.2805, -0.0972,  0.2279, -0.8800, -0.6960],
        [ 0.9936, -0.9996, -0.5091, -0.9021, -0.9943],
        [ 0.8079, -0.9193, -0.5482, -0.4247, -0.9897]],
       grad_fn=<TanhBackward0>)]


In [53]:
import torchvision
import torchvision.transforms as transforms

In [54]:
BATCH_SIZE = 64

# list all transformations
transform = transforms.Compose(
    [transforms.ToTensor()])
# download and load training dataset
trainset = torchvision.datasets.MNIST(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE,
                                          shuffle=True, num_workers=0)
# download and load testing dataset
testset = torchvision.datasets.MNIST(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=BATCH_SIZE,
                                         shuffle=False, num_workers=0)

100%|██████████| 9.91M/9.91M [00:04<00:00, 2.34MB/s]


RuntimeError: Error downloading train-labels-idx1-ubyte.gz:
Tried https://ossci-datasets.s3.amazonaws.com/mnist/, got:
<urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self signed certificate in certificate chain (_ssl.c:1007)>
Tried http://yann.lecun.com/exdb/mnist/, got:
HTTP Error 404: Not Found


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
# functions to show an image
def imshow(img):
    #img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
# get some random training images
data_iter = iter(trainloader)
images, labels = next(data_iter)
# show images
imshow(torchvision.utils.make_grid(images)

SyntaxError: incomplete input (3795983733.py, line 12)